In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import os
import csv

import talos
from datetime import datetime
from keras.activations import relu, elu, linear, sigmoid

#from tensorflow import keras
from keras import layers
from keras import models
from keras.activations import *
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, History

from data_toolsFunctional import load_preprocessed, dataPrep, nameModel


#simPrefix = os.getcwd()+'\\simdata'


In [2]:
simPrefix = 'C:\\Users\\mkrit\\Documents\\sim_data'
## location for sim files
x, y = load_preprocessed(simPrefix, 'train') 
#nan meants not a number
params = {#'activation1':['relu', 'sigmoid'],
     'first_hidden_layer': [128, 64, 32],
     'second_hidden_layer': [128, 64, 32],
     'No_of_Dense_Layers': [2,3,4],
     'Conv_filters':[60, 40],
     'epochs': [2]}

#nan meants not a number

Percentage of events with a NaN: 2.68


In [3]:
print(x.shape)
##NumPy arrays have an attribute called shape that returns a tuple with each index having the number of corresponding elements.

print(y.keys()) ##don't understand what all the dictionary keys are.

##The keys() method returns a view object. The view object contains the keys of the dictionary, as a list.
##Dictionaries are used to store data values in key:value pairs.

    # each station has 2 tanks, each tank has 2 DOMs (high/log gain)\n",
    # each tank measures charge and time\n",
    # each station gives 2 charges and 2 times, 4 total pieces of data per station\n",
    # stations arranged in 10x10 square lattice, 2 corners of square unused\n",
    # charge measured in VEM, vertical equivalent muon\n",
    # 'dir' is true direction, rest of dir are reconstruted by simulations\n",
    # 'plane_dir' assumes shower is flat plane\n",
    # 'laputop_dir' performs likelihood analysis\n",
    # 'small_dir' compromises between plane and laputop"

(549773, 10, 10, 4)
dict_keys(['comp', 'energy', 'dir', 'plane_dir', 'laputop_dir', 'small_dir'])


In [4]:
# Name for model
#key = 'Functional'
#i = 0
#while(os.path.exists('model/model_{}.h5'.format(key+str(i)))):
#    i = i + 1
#key = key+str(i)
#numepochs = 6
# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1

prep = {'q':None, 't':False, 'normed':True, 'reco':'small', 'cosz':True}
#print(y['small_dir'].shape)
#print(y['plane_dir'].shape)
#print(x.shape)
# Establishing arrays to be trained on
x_i = dataPrep(x, y, **prep)

In [5]:
# 85/15 split for training/testing
##
energy = y['energy']
comp = y['comp']

theta, phi = y['dir'].transpose() ##dir shows all attributes of an object
## For an array a with two axes, transpose(a) gives the matrix transpose.

nevents = len(x_i[1])
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)


In [6]:
#x_i[0][trainCut]
#x_i[1][trainCut]
#x_i[0][testCut]
#x_i[1][testCut]

In [7]:
# Create model
## CNN Deep Learning algorithm which can take in an input image, assign importance
##     (learnable weights and biases) to various aspects/objects in the image and be able to differentiate one from the other. 

## Ex Talos Model for syntax reference
    #input_layer = Input(shape=(42,))
    #hidden_layer1 = Dense(params['neurons'], activation=params['activation'])(input_layer)
    #hidden_layer2 = Dense(params['neurons'], activation=params['activation'])(hidden_layer1)
    #hidden_layer3 = Dense(params['neurons'], activation=params['activation'])(hidden_layer2)

    #p = {'activation1':['relu', 'elu'],
     #'activation2':['relu', 'elu'],
     #'activation3':['relu', 'elu'],
     #'first_dense_layer': [128, 64, 32],
     #'second_hidden_layer': [128, 64, 32],
     #'No_of_Dense_Layers': [2,3,4],
     #'Conv_filters1':[60,40,80,120],
     #'Conv_filters2':[60,40,80,120],
     #'Conv_filters3':[60,40,80,120],
     #'kernal_Size':[(3,3),(5,5)],
     #'epochs': [10, 15]}

    
#conv1_layer = layers.Conv2D(128,kernel_size=3,activation='relu')(charge_input)
#conv2_layer = layers.Conv2D(34,kernel_size=3,activation='relu')(conv1_layer)
#conv3_layer = layers.Conv2D(4,kernel_size=3,activation='relu')(conv2_layer)
#flat_layer = layers.Flatten()(conv2_layer)

def Functional( x_train, y_train, x_val, y_val, params):
    charge_input=keras.Input(shape=(10,10,2,))

    conv1_layer = layers.Conv2D(128,kernel_size=3,activation='relu')(charge_input)
    conv2_layer = layers.Conv2D(34,kernel_size=3,activation='relu')(conv1_layer)
    conv3_layer = layers.Conv2D(4,kernel_size=3,activation='relu')(conv2_layer)
    flat_layer = layers.Flatten()(conv3_layer)

# inputting zenith here
    zenith_input=keras.Input(shape=(1,))
    print(zenith_input.shape)
    
#A concatenation layer takes inputs and concatenates them along a specified dimension. 
#The Concat layer is a utility layer that concatenates its multiple input blobs to one single output blob.
    concat_layer = layers.concatenate([flat_layer,zenith_input])

#output = layers.Dense(1)(concat_layer)

    dense1_layer = layers.Dense(params['Conv_filters'],activation ='relu')(concat_layer)
    dropout_layer1 = layers.Dropout(0.5, input_shape = (2,))(dense1_layer)
    dense2_layer = layers.Dense(params['Conv_filters'],activation ='relu')(dense1_layer)
    dropout_layer2 = layers.Dropout(0.5, input_shape = (2,))(dense2_layer)
    output = layers.Dense(1)(dense2_layer)

    model = models.Model(inputs=[charge_input,zenith_input],outputs=output)
    model.summary()

#compiling the model
    model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])
   
    
    history = model.fit(x=x_train, y=y_train, validation_data=[x_val,y_val], epochs=params['epochs'])
    
    return history, model
    

In [8]:


scan_object = talos.Scan(x=[x_i[0][trainCut], x_i[1][trainCut]], y=energy[trainCut], x_val=[x_i[0][testCut], x_i[1][testCut]], y_val=energy[testCut], model=Functional, params=params, experiment_name='talos', fraction_limit=0.1)


#Training

#We call fit(), which will train the model by slicing the data into "batches" of size batch_size, 
#and repeatedly iterating over the entire dataset for a given number of epochs.

#csv_logger = keras.callbacks.CSVLogger('models/{}'.format(key))
#early_stop = EarlyStopping()
#callbacks = [early_stop, csv_logger]
#callbacks = [csv_logger]
#history = model.fit(
 #   {"charge":x_i[0],"zenith":x_i[1].reshape(-1,1)}, temp_y, epochs=166,validation_split=0.15,callbacks=callbacks)
  


  0%|          | 0/5 [00:00<?, ?it/s]

(None, 1)
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 10, 10, 2)]  0           []                               
                                                                                                  
 conv2d (Conv2D)                (None, 8, 8, 128)    2432        ['input_1[0][0]']                
                                                                                                  
 conv2d_1 (Conv2D)              (None, 6, 6, 34)     39202       ['conv2d[0][0]']                 
                                                                                                  
 conv2d_2 (Conv2D)              (None, 4, 4, 4)      1228        ['conv2d_1[0][0]']               
                                                                                    

 20%|██        | 1/5 [10:04<40:16, 604.06s/it]

(None, 1)
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 10, 10, 2)]  0           []                               
                                                                                                  
 conv2d (Conv2D)                (None, 8, 8, 128)    2432        ['input_1[0][0]']                
                                                                                                  
 conv2d_1 (Conv2D)              (None, 6, 6, 34)     39202       ['conv2d[0][0]']                 
                                                                                                  
 conv2d_2 (Conv2D)              (None, 4, 4, 4)      1228        ['conv2d_1[0][0]']               
                                                                                    

KeyboardInterrupt: 

In [ ]:
#Save an array to a binary file in NumPy .npy format.

np.save("Functional.npy",prep)
model.save('models/model_%s.h5' % key)
#f = open("results.txt", "a")
#now = datetime.now()
#f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    ##now.strftime("%m/%d/%Y %H:%M:%S"),
    #key,
    #len(history.history['loss']),
    #history.history['loss'][len(history.history['loss'])-1],
    #history.history['val_loss'][len(history.history['loss'])-1]
#))
#f.close()

In [ ]:
# get the name of the experiment log
!ls -lhtr talos
#get the name of the .csv file you want to use, and use it as part of the input for experiment_log in the next step.